# 带有负载的有损传输线上的损耗

在处理射频功率时，例如在无线电、工业或科学应用中，一个常见的问题是如何正确处理不可避免的射频损耗，以避免电缆和组件过热。## 匹配负载在本例中，我们将使用 `scikit-rf` 来评估一根 50 欧姆、长 20 米、RG-8 电缆（VF=0.84）的损耗，该电缆在 13.56 MHz 频率下连接了一个电阻性负载 $R_L=50\Omega$。电缆的损耗估计为 1.483 dB/100 米，源功率为 400W。

首先，导入常用的 Python 模块：

In [ ]:
%matplotlib inline

In [ ]:
import numpy as np

import skrf as rf
from skrf.constants import c
from skrf.mathFunctions import db_2_np, db_per_100feet_2_db_per_100meter, feet_2_meter, mag_2_db10
from skrf.tlineFunctions import (
    Gamma0_2_swr,
    input_impedance_at_theta,
    voltage_current_propagation,
    zl_2_Gamma0,
    zl_2_Gamma_in,
    zl_2_total_loss,
    zl_2_zin,
)

In [ ]:
rf.stylely()

让我们定义问题中的常数：

In [ ]:
Pin = 400  # W
z0 = 50 # Ohm
freq = rf.Frequency(13.56, npoints=1, unit='MHz')
VF = 0.84
RL = 50  # Ohm
L = 20  # m

传输线的传播常数 $\gamma=\alpha+j\beta$ 为：

In [ ]:
alpha = db_2_np(1.483/100)  # Np/m
beta = freq.w/c/VF
gamma = alpha + 1j*beta

匹配线路的损耗（或功率衰减），$a=e^{2\alpha L}$，为：

In [ ]:
a = np.exp(2*alpha*L)  # also simply 2.84/100*20
print('Matched line loss: a=', mag_2_db10(a), 'dB')

如果传输线连接了一个匹配的负载，即 $R_L = 50\Omega$，那么总的线路损耗为 $a$。因此，电缆中的功率损耗将为：

In [ ]:
print('(Forward) Power delivered to the  load:', Pin/a, 'W')
print('Power lost in the cable:', Pin *( 1 - 1/a), 'W')

这也可以通过 `scikit-rf` 中的传输线函数 `zl_2_total_loss` 来验证：

In [ ]:
a_skrf = zl_2_total_loss(z0, zl=RL, theta=gamma*L)
print('Power lost in the cable:',
      Pin * (1 - 1/a_skrf), 'W')

评估电路中总损耗功率的另一种方法是评估功率表达式：$$P_{delivered} = \frac{1}{2} \Re \left[ V I^* \right]$$其中，$V$ 和 $I$ 分别是（峰值）总电压和总电流。可以使用传输线函数 `voltage_current_propagation` 来计算它们：

In [ ]:
# reflection coefficient and input impedance
Gamma_in = zl_2_Gamma_in(z0, RL, theta=gamma*L)
Z_in = zl_2_zin(z0, RL, theta=gamma*L)

# voltage and current at the line input as a function of source power
V_in = np.sqrt(2*z0*Pin)*(1 + Gamma_in)
I_in = V_in/Z_in

# voltage and current at z=L
V,I = voltage_current_propagation(V_in, I_in, z0, gamma*L)
P_delivered = 1/2 * np.real(V * np.conj(I))
print('Power delivered to the load: ', P_delivered, 'W')
print('Power dissipated in the cable: ',Pin - P_delivered, 'W')

## 不匹配的负载

但是，如果负载与线路的特性阻抗 $z_0$ 不完全匹配，例如 $R_L=200 + 30j\Omega$，反射波会引起额外的损耗。由此负载引起的反射系数 $\Gamma_{load}$ 为：

In [ ]:
z0 = 50
ZL = 200 - 30j
Gamma_load = zl_2_Gamma0(z0, ZL)
print('|Gamma_load|=', np.abs(Gamma_load))

而从传输线的输入端看到的反射系数 $\Gamma_{in}$ 为：

In [ ]:
Gamma_in = zl_2_Gamma_in(z0, ZL, theta=gamma*L)
SWR = Gamma0_2_swr(zl_2_Gamma_in(z0, ZL, theta=gamma*L))
print('|Gamma_in|=', np.abs(Gamma_in), '(SWR=', SWR,')')

由于驻波比引起的总损耗（单位：dB）通常表示为：$$a_{[dB]} + 10 \log_{10} \frac{1 - |\Gamma_{in}|^2}{1 - |\Gamma_{load}|^2}$$

In [ ]:
10*np.log10(a) + 10*np.log10((1 - np.abs(Gamma_in)**2)/(1 - np.abs(Gamma_load)**2))

但是，只有当以下任一条件满足时，此表达式才是正确的：- (i) 线路的特性阻抗为实数（无失真线路）- (ii) 反射系数为实数（即，实数 $Z_L/Z_0$）[1]。第一个条件在这里满足，但下一个部分将不满足。无论如何，`scikit-rf` 传输线函数 `zl_2_total_loss` 在所有条件下都是正确的：

In [ ]:
a = zl_2_total_loss(z0, zl=ZL, theta=gamma * L)
print('Total power loss: ', mag_2_db10(a), 'dB' )
print('Delivered power:', Pin/a, 'W')
print('The total power loss is the cable:', Pin*(1 - 1/a), 'W')

In [ ]:
# reflection coefficient and input impedance
Gamma_in = zl_2_Gamma_in(z0, ZL, theta=gamma*L)
Z_in = zl_2_zin(z0, ZL, theta=gamma*L)

# voltage and current at the line input as a function of source power
V_in = np.sqrt(2*z0*Pin)*(1 + Gamma_in)
I_in = V_in/Z_in

# voltage and current at z=L
V,I = voltage_current_propagation(V_in, I_in, z0, gamma*L)
P_delivered = 1/2 * np.real(V * np.conj(I))
print('Power delivered to the load: ', P_delivered, 'W')
print('Power dissipated in the cable: ',Pin - P_delivered, 'W')

In [ ]:
Gamma0_2_swr(Gamma_in)

In [ ]:
10*np.log10(P_delivered/Pin)

## 一个更高级的示例本示例重现了参考资料 [1] 中给出的示例。假设我们有一根同轴电缆（[Wireman #551，阻抗为 450 欧姆](https://thewireman.com/antennap.html)），其一端连接了一个复数负载 $Z_L=R_L + jX_L$，参数如下：- 电缆长度：100 英尺- 频率：1.83 MHz- [衰减常数](https://en.wikipedia.org/wiki/Propagation_constant#Attenuation_constant)：$\alpha=$ 0.095 dB/100 英尺- 同轴电缆相对介电常数：$\epsilon_r=1.194$（[速度因子](https://en.wikipedia.org/wiki/Velocity_factor) VF=0.915）- 特性阻抗的实部：$R_0 = \Re \left[Z_0\right]$=402.75 欧姆- 负载电阻：$R_L$ = 4.5 欧姆- 负载电抗：$X_L$ = -1673 欧姆

In [ ]:
Z_L = 4.5 - 1673j
R_0 = 402.75
freq = rf.Frequency(1.83, npoints=1, unit='MHz')
VF = 0.915
L = feet_2_meter(100)

首先，我们可以从问题参数中推导出传播常数 $\gamma=\alpha+j\beta$，其中 $\beta=\frac{\omega}{c }\sqrt{\epsilon_r}$。

In [ ]:
alpha = db_2_np(db_per_100feet_2_db_per_100meter(0.095)/100)
beta = freq.w/c/VF
gamma = alpha + 1j*beta
print(gamma)

然而，传输线的特性电抗并未在问题参数中给出，因此需要对其进行确定。 可以通过高频低损耗近似 [1] 来近似计算：$$Z_0 = R_0 + j X_0 \approx R_0 - j \frac{\alpha}{\beta}R_0$$即，$$X_0 \approx - \frac{\alpha}{\beta}R_0$$

In [ ]:
X_0 = -alpha/beta*R_0
Z_0 = R_0 + 1j*X_0
print('X_0=', X_0)

现在我们已经获得了线路的特性阻抗和传播常数，就可以推导出反射系数、输入阻抗和总损耗：

In [ ]:
print('Gamma at load:', np.abs(zl_2_Gamma0(Z_0, Z_L)))
print('Gamma at input:', np.abs(zl_2_Gamma_in(Z_0, Z_L, theta=gamma*L)))
print('SWR at load:', Gamma0_2_swr(zl_2_Gamma0(Z_0, Z_L)))

print('SWR at input:', Gamma0_2_swr(zl_2_Gamma_in(Z_0, Z_L, theta=gamma*L)))
print('Input impedance:', input_impedance_at_theta(Z_0, Z_L, theta=gamma*L ), 'Ohm')

total_loss_db = mag_2_db10(np.abs(zl_2_total_loss(z0=Z_0, zl=Z_L, theta=gamma*L)))
print('Total loss:', total_loss_db, 'dB')

这些结果与参考文献[1]中呈现的结果相符。

## References - [1] Steve Stearns (K6OIK), A Transmission Line Power Paradox and Its Resolution, ARRL Pacificon Antenna Seminar, Santa Clara, CA          October 10-12, 2014: https://www.fars.k6ya.org/docs/K6OIK-A_Transmission_Line_Power_Paradox_and_Its_Resolution.pdf 